In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("D:/churn-prediction-project/data/raw/churn_raw.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Shape : {df.shape}")

Shape : (7032, 21)


In [5]:
# 1. Ratio charges totales / ancienneté (valeur moyenne par mois)
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)

# 2. Écart entre charge mensuelle actuelle et moyenne historique
df['spend_drift'] = df['MonthlyCharges'] - df['avg_monthly_spend']

# 3. Nombre de services souscrits
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport',
                'StreamingTV', 'StreamingMovies']
df['nb_services'] = (df[service_cols] == 'Yes').sum(axis=1)

# 4. Client sans aucun service de sécurité/support
df['no_protection'] = (
    (df['OnlineSecurity'] == 'No') &
    (df['TechSupport'] == 'No') &
    (df['DeviceProtection'] == 'No')
).astype(int)

# 5. Segments d'ancienneté
df['tenure_segment'] = pd.cut(df['tenure'],
    bins=[0, 12, 36, 60, 100],
    labels=['Nouveau', 'Intermédiaire', 'Fidèle', 'Très fidèle'])

# 6. Segment de valeur
df['value_segment'] = pd.cut(df['MonthlyCharges'],
    bins=[0, 35, 65, 200],
    labels=['Petit compte', 'Compte moyen', 'Grand compte'])

print("✅ Features créées")
df[['avg_monthly_spend', 'spend_drift', 'nb_services',
    'no_protection', 'tenure_segment', 'value_segment']].head(10)

✅ Features créées


,avg_monthly_spend,spend_drift,nb_services,no_protection,tenure_segment,value_segment
0,14.925000,14.925000,1,1,Nouveau,Petit compte
1,53.985714,2.964286,3,0,Intermédiaire,Compte moyen
2,36.050000,17.800000,3,0,Nouveau,Compte moyen
3,40.016304,2.283696,3,0,Fidèle,Compte moyen
4,50.550000,20.150000,1,1,Nouveau,Grand compte
5,91.166667,8.483333,5,0,Nouveau,Grand compte
6,84.756522,4.343478,4,1,Intermédiaire,Grand compte
7,27.445455,2.304545,1,0,Nouveau,Petit compte
8,105.036207,-0.236207,6,0,Intermédiaire,Grand compte
9,55.364286,0.785714,3,0,Très fidèle,Compte moyen


In [7]:
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

# Colonnes à encoder
cat_cols = df_model.select_dtypes('object').columns.tolist()
cat_cols += ['tenure_segment', 'value_segment']

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Supprimer l'ID client (inutile pour le ML)
df_model = df_model.drop(columns=['customerID'])

print(f"Shape final : {df_model.shape}")
print(f"Types :\n{df_model.dtypes.value_counts()}")

Shape final : (7032, 26)
Types :
int32      18
int64       4
float64     4
Name: count, dtype: int64


In [11]:
df_model.to_csv("D:/churn-prediction-project/data/processed/churn_features.csv", index=False)
print("✅ Dataset sauvegardé dans data/processed/churn_features.csv")
print(f"\nColonnes finales ({len(df_model.columns)}) :")
print(df_model.columns.tolist())

✅ Dataset sauvegardé dans data/processed/churn_features.csv

Colonnes finales (26) :
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'avg_monthly_spend', 'spend_drift', 'nb_services', 'no_protection', 'tenure_segment', 'value_segment']
